# FSL: Structural MRI Pipeline

Brain extraction (BET) -> tissue segmentation (FAST) -> linear registration to MNI (FLIRT) -> nonlinear warp (FNIRT) -> subcortical segmentation (FIRST). Uses real T1w from ds000114.

## Prerequisites

- **Python 3.9+**
- **FSL 6.0+** installed and configured (https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/FslInstallation)
- `FSLDIR` environment variable should be set (the notebook falls back to `/usr/local/fsl` if unset)
- Basic understanding of structural MRI preprocessing (skull stripping, tissue segmentation, spatial normalization)

## Setup

### Data Download

This notebook requires a T1-weighted structural MRI. You can use any T1w NIfTI file, or download
a sample from OpenNeuro using datalad:

```bash
pip install datalad
datalad install https://github.com/OpenNeuroDatasets/ds000114
cd ds000114
datalad get sub-01/ses-test/anat/sub-01_ses-test_T1w.nii.gz
```

Alternatively, the setup cell below downloads a sample brain using nilearn.

In [ ]:
# Install required packages
!pip install nibabel nilearn matplotlib numpy

In [ ]:
import subprocess, os, nibabel as nib, numpy as np
import nilearn.plotting as nlp
import matplotlib.pyplot as plt

# Set up FSL environment -- use FSLDIR from environment, with fallback
FSLDIR = os.environ.get('FSLDIR', '/usr/local/fsl')
os.environ['FSLDIR'] = FSLDIR
os.environ['FSLOUTPUTTYPE'] = 'NIFTI_GZ'
os.environ['PATH'] = f"{FSLDIR}/bin:" + os.environ['PATH']

# Output directory (portable, relative path)
OUT = 'output/fsl_structural'
os.makedirs(OUT, exist_ok=True)

# MNI template from FSL's standard data directory
MNI = os.path.join(FSLDIR, 'data', 'standard', 'MNI152_T1_1mm_brain.nii.gz')

# --- Subject T1w: choose one of the options below ---

# Option A: If you downloaded ds000114 via datalad, set the path:
# T1 = 'ds000114/sub-01/ses-test/anat/sub-01_ses-test_T1w.nii.gz'

# Option B: Download a sample brain using nilearn
from nilearn.datasets import fetch_oasis_vbm
oasis = fetch_oasis_vbm(n_subjects=1)
T1 = oasis.gray_matter_maps[0]

print('T1w:', nib.load(T1).shape)
ver = subprocess.run(['flirt', '-version'], capture_output=True, text=True)
print('FSL version:', ver.stdout.strip() or ver.stderr.strip() or 'check PATH')

In [ ]:
# Step 1: Brain extraction with BET
brain = f'{OUT}/sub-01_brain'
result = subprocess.run(['bet', T1, brain, '-R', '-f', '0.5', '-g', '0', '-m'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('BET stderr:', result.stderr[:500])
    raise RuntimeError('BET failed -- check FSL PATH above')
print(result.stdout or 'BET complete')

brain_nii = f'{brain}.nii.gz'
if not os.path.exists(brain_nii):
    raise FileNotFoundError(f'Expected BET output not found: {brain_nii}')

# Visualize skull strip result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
t1_img = nib.load(T1)
brain_img = nib.load(brain_nii)
sl = t1_img.shape[2] // 2
axes[0].imshow(t1_img.get_fdata()[:, :, sl].T, cmap='gray', origin='lower'); axes[0].set_title('Original T1w')
axes[1].imshow(brain_img.get_fdata()[:, :, sl].T, cmap='gray', origin='lower'); axes[1].set_title('Brain extracted (BET)')
plt.tight_layout(); plt.show()

In [ ]:
# Step 2: Tissue segmentation with FAST (GM / WM / CSF)
result = subprocess.run(['fast', '-t', '1', '-n', '3', '-H', '0.1', '-I', '4', '-l', '20.0',
                          '-g', '--nopve', '-o', f'{OUT}/sub-01_seg', f'{brain}.nii.gz'],
                         capture_output=True, text=True)
print('FAST complete')

# Plot tissue classes
import glob
segs = sorted(glob.glob(f'{OUT}/sub-01_seg_pve_*.nii.gz'))
labels = ['CSF', 'Gray Matter', 'White Matter']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, seg, label) in enumerate(zip(axes, segs, labels)):
    img = nib.load(seg).get_fdata()
    ax.imshow(img[:, :, img.shape[2]//2].T, cmap='hot', origin='lower')
    ax.set_title(f'FAST: {label}')
plt.suptitle('Tissue segmentation (FAST)'); plt.tight_layout(); plt.show()

In [ ]:
# Step 3: Linear registration to MNI (FLIRT)
mni_xfm = f'{OUT}/sub-01_to_MNI_affine.mat'
mni_brain = f'{OUT}/sub-01_brain_MNI_affine.nii.gz'
result = subprocess.run(['flirt', '-in', f'{brain}.nii.gz', '-ref', MNI,
                          '-out', mni_brain, '-omat', mni_xfm,
                          '-bins', '256', '-cost', 'corratio', '-dof', '12'],
                         capture_output=True, text=True)
print('FLIRT complete. Matrix saved to:', mni_xfm)
nlp.plot_anat(mni_brain, title='Sub-01 registered to MNI152 (FLIRT affine)')
plt.show()

In [ ]:
# Step 4: Non-linear warp to MNI (FNIRT)
mni_fnirt = f'{OUT}/sub-01_brain_MNI_fnirt.nii.gz'
warp_coef = f'{OUT}/sub-01_warp_coef.nii.gz'
result = subprocess.run(['fnirt', f'--in={brain}.nii.gz', f'--aff={mni_xfm}',
                          f'--cout={warp_coef}', f'--iout={mni_fnirt}',
                          '--config=T1_2_MNI152_2mm'],
                         capture_output=True, text=True)
print('FNIRT complete')
nlp.plot_anat(mni_fnirt, title='Sub-01 -> MNI152 (FNIRT nonlinear)')
plt.show()

## References

- Smith, S. M. (2002). Fast robust automated brain extraction. *Human Brain Mapping*, 17(3), 143-155. https://doi.org/10.1002/hbm.10062
- Zhang, Y., Brady, M., & Smith, S. (2001). Segmentation of brain MR images through a hidden Markov random field model and the expectation-maximization algorithm. *IEEE Transactions on Medical Imaging*, 20(1), 45-57. https://doi.org/10.1109/42.906424
- Jenkinson, M., Bannister, P., Brady, M., & Smith, S. (2002). Improved optimization for the robust and accurate linear registration and motion correction of brain images. *NeuroImage*, 17(2), 825-841. https://doi.org/10.1006/nimg.2002.1132
- Andersson, J. L. R., Jenkinson, M., & Smith, S. (2007). Non-linear registration, aka spatial normalisation. *FMRIB Technical Report TR07JA2*. https://www.fmrib.ox.ac.uk/datasets/techrep/tr07ja2/tr07ja2.pdf
- Jenkinson, M., Beckmann, C. F., Behrens, T. E. J., Woolrich, M. W., & Smith, S. M. (2012). FSL. *NeuroImage*, 62(2), 782-790. https://doi.org/10.1016/j.neuroimage.2011.09.015
- FSL documentation: https://fsl.fmrib.ox.ac.uk/fsl/fslwiki